# AIBO（aibo_v8 リポジトリ）— Google Colab · Nunchaku 経路

- **`01_config.py` など数字始まりファイル**は **`load_module_from_file`** で読み込みます。第1引数の **モジュール名は `01_config` のようにリポ内の `import_module("01_config")` と同じ文字列**にし、`sys.modules["01_config"]` に載せます（`config_module` 等だと内部 import が壊れます）。
- **エラーで `import_module("01_config")` が出る場合**: Colab が古いノートを開いています。[GitHub の `aibo_v7_colab.ipynb`](https://github.com/miya390831-a11y/aibo_v8/blob/master/aibo_v7_colab.ipynb) を開き直すか、Drive のコピーを最新に差し替えてください。
- **ソースはブランチ `colab-stable` をクローンします**（RunPod / Docker 追加前の Colab 向けスナップショット、`393942a` 系）。`master` の RunPod 改変は含みません。
- 実行は **`02_colab_setup.py` の `ColabBootstrap` + `07_main.run()`**（`01_config.RuntimeStrategy` に従う Colab 経路）です。
- **Colab の A100 40GB** では `01_config.py` の `RuntimeStrategy.detect()` が通常 **`a100_40gb_nunchaku`** を選びます（VRAM ≥ 38GB かつ GPU 名が A6000 系ルールに当たらない場合）。**A100 80GB** では `a100_80gb_bf16` になります。
- **`!python 02_colab_setup.py` 単体**はリポ内で **診断・wheel 候補表示のみ**（`bootstrap.run()` 未実行）なので、下のセルでは **`ColabBootstrap(...).run()` を必ず続けて実行**します。
- **`pip install nunchaku` は非推奨**（PyPI の別物パッケージと衝突することがあります）。**公式 wheel は `ColabBootstrap` が入れます。**

In [ ]:
!git clone -b colab-stable https://github.com/miya390831-a11y/aibo_v8.git
%cd aibo_v8

In [ ]:
!nvidia-smi

In [ ]:
# 依存・Drive マウント・公式 Nunchaku wheel・PuLID 取得など（Colab 本番セットアップ）
# 参考: !pip install -q nunchaku  ← PyPI 偽パッケージのリスクがあるため省略推奨

import importlib.util
import os
import subprocess
import sys

ROOT = os.path.abspath(".")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)


def load_module_from_file(name, path):
    """name はリポジトリ内の import と同じ（例: '01_config'）→ sys.modules[name] に登録"""
    path = os.path.abspath(path)
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    assert spec.loader is not None
    spec.loader.exec_module(mod)
    return mod


# 先に 01_config を載せておくと、以降の import_module("01_config") と整合しやすい
load_module_from_file("01_config", "01_config.py")

# リポジトリ同梱の診断（wheel URL 候補の表示など）。フルセットアップは続く bootstrap.run()。
subprocess.run([sys.executable, "02_colab_setup.py"], cwd=".", check=False)

cfg_mod = sys.modules["01_config"]
setup_mod = load_module_from_file("02_colab_setup", "02_colab_setup.py")
sys_cfg = cfg_mod.SystemConfig()
bootstrap = setup_mod.ColabBootstrap(sys_cfg)
bootstrap.run()

In [ ]:
from google.colab import userdata
import os

_hf = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = _hf
os.environ["HUGGINGFACE_HUB_TOKEN"] = _hf

In [ ]:
# 全 Phase（Pipeline build · Gradio UI 起動など）
# 注: !python 07_main.py はリポジトリ内の「軽量診断」のみ。
import importlib.util
import os
import sys

ROOT = os.path.abspath(".")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)


def load_module_from_file(name, path):
    path = os.path.abspath(path)
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    assert spec.loader is not None
    spec.loader.exec_module(mod)
    return mod


main_mod = load_module_from_file("07_main", "07_main.py")
main_mod.run()

## 確認: 戦略が `a100_40gb_nunchaku` か（A100 40GB 想定）
セルがまだ GPU ランタイム上であれば、次を実行してください。**80GB では `a100_80gb_bf16` が正しい挙動**です。

In [ ]:
import importlib.util
import os
import sys

ROOT = os.path.abspath(".")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)


def load_module_from_file(name, path):
    path = os.path.abspath(path)
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    assert spec.loader is not None
    spec.loader.exec_module(mod)
    return mod


if "01_config" not in sys.modules:
    load_module_from_file("01_config", "01_config.py")
cfg_mod = sys.modules["01_config"]
r = cfg_mod.RuntimeStrategy.detect()
print("strategy:", r.kind.value)
print("vram_gb:", round(r.vram_gb, 2), "gpu:", r.gpu_name)
print("nunchaku_available:", r.nunchaku_available)